In [1]:
import os

os.environ[
    "TF_USE_LEGACY_KERAS"
] = "1"

In [4]:
!pip install -q tensorflow-recommenders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.9/108.9 kB 2.8 MB/s eta 0:00:00a 0:00:01


In [5]:
import pandas as pd
import pickle

import tensorflow as tf
import tensorflow_recommenders as tfrs

import numpy as np

In [6]:
training_df = pd.read_parquet(
    "/kaggle/input/datasets/selinparmar/train-ds/training_dataset.parquet"
)

with open(
    "/kaggle/input/datasets/selinparmar/feature-vocab/feature_vocab.pkl",
    "rb"
) as file:

    feature_vocab = pickle.load(file)

In [7]:
training_df = (
    training_df
    .sample(
        200000,
        random_state=42
    )
)

print(
    training_df.shape
)

(200000, 16)


In [8]:
training_df[
    'customer_id'
] = (
    training_df[
        'customer_id'
    ].astype(str)
)

training_df[
    'article_id'
] = (
    training_df[
        'article_id'
    ].astype(str)
)

In [9]:
user_vocab = (
    feature_vocab[
        'user_vocab'
    ]
)

favorite_product_vocab = (
    feature_vocab[
        'favorite_product_vocab'
    ]
)

item_vocab = (
    feature_vocab[
        'item_vocab'
    ]
)

product_type_vocab = (
    feature_vocab[
        'product_type_vocab'
    ]
)

season_vocab = (
    feature_vocab[
        'season_vocab'
    ]
)

In [10]:
class QueryTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Customer ID embedding
        self.customer_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=user_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(user_vocab) + 1,
                32
            )
        ])

        # Favorite product type embedding
        self.favorite_product_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                favorite_product_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    favorite_product_vocab
                ) + 1,
                16
            )
        ])

        # Season embedding
        self.season_embedding = tf.keras.Sequential([

            tf.keras.layers.StringLookup(
                vocabulary=
                season_vocab,
                mask_token=None
            ),

            tf.keras.layers.Embedding(
                len(
                    season_vocab
                ) + 1,
                8
            )
        ])

        # Dense network
        self.dense_layers = tf.keras.Sequential([

            tf.keras.layers.Dense(
                128,
                activation='relu'
            ),

            tf.keras.layers.Dense(64)
        ])

    def call(self, features):

        customer_vector = (
            self.customer_embedding(
                features['customer_id']
            )
        )

        favorite_product_vector = (
            self.favorite_product_embedding(
                features[
                    'favorite_product_type'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features['season']
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features['age'],
                tf.float32
            ),

            tf.cast(
                features[
                    'purchase_frequency'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'avg_spend'
                ],
                tf.float32
            ),

            tf.cast(
                features[
                    'repeat_purchase_ratio'
                ],
                tf.float32
            )

        ], axis=1)

        return self.dense_layers(
            tf.concat([

                customer_vector,
                favorite_product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

In [11]:
class CandidateTower(
    tf.keras.Model
):

    def __init__(self):

        super().__init__()

        # Article embedding
        self.article_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=item_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(item_vocab) + 1,
                    32
                )
            ])
        )

        # Product type embedding
        self.product_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    product_type_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        product_type_vocab
                    ) + 1,
                    16
                )
            ])
        )

        # Season embedding
        self.season_embedding = (
            tf.keras.Sequential([

                tf.keras.layers.StringLookup(
                    vocabulary=
                    season_vocab,
                    mask_token=None
                ),

                tf.keras.layers.Embedding(
                    len(
                        season_vocab
                    ) + 1,
                    8
                )
            ])
        )

        # Dense network
        self.dense_layers = (
            tf.keras.Sequential([

                tf.keras.layers.Dense(
                    128,
                    activation='relu'
                ),

                tf.keras.layers.Dense(
                    64
                )
            ])
        )

    def call(
        self,
        features
    ):

        article_vector = (
            self.article_embedding(
                features[
                    'article_id'
                ]
            )
        )

        product_vector = (
            self.product_embedding(
                features[
                    'product_type_name'
                ]
            )
        )

        season_vector = (
            self.season_embedding(
                features[
                    'season'
                ]
            )
        )

        numeric_features = tf.stack([

            tf.cast(
                features[
                    'purchase_count'
                ],
                tf.float32
            )

        ], axis=1)

        combined_features = (
            tf.concat([

                article_vector,
                product_vector,
                season_vector,
                numeric_features

            ], axis=1)
        )

        return (
            self.dense_layers(
                combined_features
            )
        )

In [12]:
#candidate dataset
candidate_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        {
            "article_id":
                training_df[
                    "article_id"
                ],

            "product_type_name":
                training_df[
                    "product_type_name"
                ],

            "season":
                training_df[
                    "season"
                ],

            "purchase_count":
                training_df[
                    "purchase_count"
                ]
                .astype(float)
        }
    )
    .batch(256)
)

2026-05-25 15:55:16.084871: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [13]:
class TwoTowerModel(
    tfrs.models.Model
):

    def __init__(self):

        super().__init__()

        self.query_model = (
            QueryTower()
        )

        self.candidate_model = (
            CandidateTower()
        )

        self.task = (
            tfrs.tasks.Retrieval(
                metrics=
                tfrs.metrics.FactorizedTopK(

                    candidates=
                    candidate_dataset.map(
                        self.candidate_model
                    )
                )
            )
        )

    def compute_loss(
        self,
        features,
        training=False
    ):

        query_embeddings = (
            self.query_model(
                features
            )
        )

        candidate_embeddings = (
            self.candidate_model(
                features
            )
        )

        return self.task(
            query_embeddings,
            candidate_embeddings
        )

In [14]:
model = (
    TwoTowerModel()
)

model.compile(
    optimizer=
    tf.keras.optimizers.Adagrad(
        learning_rate=0.1
    )
)

In [15]:
dummy_features = {

    key:
    tf.constant(
        training_df[
            key
        ].head(1)
    )

    for key in
    training_df.columns
}

model.query_model(
    dummy_features
)

model.candidate_model(
    dummy_features
)

print(
    "Model built!"
)

Model built!


In [16]:
model.load_weights(
    "/kaggle/input/datasets/selinparmar/two-tower-weights-model-sample/two_tower_weights.weights.h5"
)

print(
    "Final model loaded!"
)

Final model loaded!


In [17]:
article_lookup = (

    training_df[

        [
            'article_id',
            'product_type_name'
        ]

    ]

    .drop_duplicates()
)

In [18]:
sample_customer = (
    training_df[
        'customer_id'
    ]
    .iloc[0]
)

sample_customer

'c18722c1ae2badf00e03179e1d55af5825de51a4d79dcb11e340391417c465f4'

In [19]:
customer_features = (

    training_df[

        training_df[
            'customer_id'
        ]
        ==
        sample_customer

    ]

    .iloc[0]
)

In [20]:
query_input = {

    'customer_id':
        tf.constant(
            [
                customer_features[
                    'customer_id'
                ]
            ]
        ),

    'favorite_product_type':
        tf.constant(
            [
                customer_features[
                    'favorite_product_type'
                ]
            ]
        ),

    'season':
        tf.constant(
            [
                customer_features[
                    'season'
                ]
            ]
        ),

    'age':
        tf.constant(
            [
                customer_features[
                    'age'
                ]
            ],
            dtype=tf.float32
        ),

    'purchase_frequency':
        tf.constant(
            [
                customer_features[
                    'purchase_frequency'
                ]
            ],
            dtype=tf.float32
        ),

    'avg_spend':
        tf.constant(
            [
                customer_features[
                    'avg_spend'
                ]
            ],
            dtype=tf.float32
        ),

    'repeat_purchase_ratio':
        tf.constant(
            [
                customer_features[
                    'repeat_purchase_ratio'
                ]
            ],
            dtype=tf.float32
        )
}

In [21]:
user_embedding = (
    model.query_model(
        query_input
    )
)

all_items = (
    training_df[
        'article_id'
    ]
    .unique()
)

scores = []

for item in all_items[:1000]:

    item_row = (
        training_df[
            training_df[
                'article_id'
            ]
            ==
            item
        ]
        .iloc[0]
    )

    item_input = {

        'article_id':
            tf.constant(
                [item]
            ),

        'product_type_name':
            tf.constant(
                [
                    item_row[
                        'product_type_name'
                    ]
                ]
            ),

        'season':
            tf.constant(
                [
                    item_row[
                        'season'
                    ]
                ]
            ),

        'purchase_count':
            tf.constant(
                [
                    item_row[
                        'purchase_count'
                    ]
                ],
                dtype=tf.float32
            )
    }

    item_embedding = (
        model.candidate_model(
            item_input
        )
    )

    score = tf.reduce_sum(

        user_embedding
        *
        item_embedding

    ).numpy()

    scores.append(
        (
            item,
            score
        )
    )

In [22]:
top_items = sorted(

    scores,

    key=lambda x: x[1],

    reverse=True

)[:10]

top_items

[('715554002', np.float32(-91.51807)),
 ('679280012', np.float32(-91.83063)),
 ('894752001', np.float32(-92.71481)),
 ('900235002', np.float32(-93.341156)),
 ('701362001', np.float32(-94.52493)),
 ('791587010', np.float32(-95.30249)),
 ('876345001', np.float32(-96.24512)),
 ('826218001', np.float32(-97.42966)),
 ('622162001', np.float32(-98.30182)),
 ('747636010', np.float32(-98.43741))]

In [23]:
recommended_products = []

for item_id, score in top_items:

    product = (
        article_lookup[

            article_lookup[
                'article_id'
            ]
            ==
            item_id

        ]

        .iloc[0]
    )

    recommended_products.append({

        "article_id":
            item_id,

        "product_type":
            product[
                'product_type_name'
            ],

        "score":
            score
    })

pd.DataFrame(
    recommended_products
)

,article_id,product_type,score
0,715554002,Blouse,-91.518066
1,679280012,Underwear bottom,-91.830627
2,894752001,Shorts,-92.714813
3,900235002,Dress,-93.341156
4,701362001,Cardigan,-94.524933
5,791587010,T-shirt,-95.302490
6,876345001,Blouse,-96.245117
7,826218001,Top,-97.429657
8,622162001,Sweater,-98.301819
9,747636010,Top,-98.437408


In [24]:
print(
    customer_features[
        'favorite_product_type'
    ]
)

print(
    customer_features[
        'season'
    ]
)

print(
    customer_features[
        'age'
    ]
)

Dress
Spring
28.0


In [25]:
sample_customer = (
    training_df[
        'customer_id'
    ]
    .iloc[50]
)

In [26]:
customer_features = (

    training_df[

        training_df[
            'customer_id'
        ]
        ==
        sample_customer

    ]

    .iloc[0]
)

In [27]:
query_input = {

    'customer_id':
        tf.constant(
            [
                customer_features[
                    'customer_id'
                ]
            ]
        ),

    'favorite_product_type':
        tf.constant(
            [
                customer_features[
                    'favorite_product_type'
                ]
            ]
        ),

    'season':
        tf.constant(
            [
                customer_features[
                    'season'
                ]
            ]
        ),

    'age':
        tf.constant(
            [
                customer_features[
                    'age'
                ]
            ],
            dtype=tf.float32
        ),

    'purchase_frequency':
        tf.constant(
            [
                customer_features[
                    'purchase_frequency'
                ]
            ],
            dtype=tf.float32
        ),

    'avg_spend':
        tf.constant(
            [
                customer_features[
                    'avg_spend'
                ]
            ],
            dtype=tf.float32
        ),

    'repeat_purchase_ratio':
        tf.constant(
            [
                customer_features[
                    'repeat_purchase_ratio'
                ]
            ],
            dtype=tf.float32
        )
}

In [28]:
user_embedding = (
    model.query_model(
        query_input
    )
)

all_items = (
    training_df[
        'article_id'
    ]
    .unique()
)

scores = []

for item in all_items[:1000]:

    item_row = (
        training_df[
            training_df[
                'article_id'
            ]
            ==
            item
        ]
        .iloc[0]
    )

    item_input = {

        'article_id':
            tf.constant(
                [item]
            ),

        'product_type_name':
            tf.constant(
                [
                    item_row[
                        'product_type_name'
                    ]
                ]
            ),

        'season':
            tf.constant(
                [
                    item_row[
                        'season'
                    ]
                ]
            ),

        'purchase_count':
            tf.constant(
                [
                    item_row[
                        'purchase_count'
                    ]
                ],
                dtype=tf.float32
            )
    }

    item_embedding = (
        model.candidate_model(
            item_input
        )
    )

    score = tf.reduce_sum(

        user_embedding
        *
        item_embedding

    ).numpy()

    scores.append(
        (
            item,
            score
        )
    )

In [29]:
top_items = sorted(

    scores,

    key=lambda x: x[1],

    reverse=True

)[:10]

top_items

[('842378001', np.float32(6.845578)),
 ('708678001', np.float32(6.126051)),
 ('772907002', np.float32(5.8232117)),
 ('651898001', np.float32(5.6772366)),
 ('693456001', np.float32(5.6189594)),
 ('873552005', np.float32(5.5682316)),
 ('921906005', np.float32(5.349017)),
 ('674649001', np.float32(5.3348036)),
 ('765545001', np.float32(5.3074894)),
 ('687635018', np.float32(5.2922163))]

In [30]:
recommended_products = []

for item_id, score in top_items:

    product = (
        article_lookup[

            article_lookup[
                'article_id'
            ]
            ==
            item_id

        ]

        .iloc[0]
    )

    recommended_products.append({

        "article_id":
            item_id,

        "product_type":
            product[
                'product_type_name'
            ],

        "score":
            score
    })

pd.DataFrame(
    recommended_products
)

,article_id,product_type,score
0,842378001,Trousers,6.845578
1,708678001,Sweater,6.126051
2,772907002,Dress,5.823212
3,651898001,T-shirt,5.677237
4,693456001,Top,5.618959
5,873552005,Vest top,5.568232
6,921906005,Dress,5.349017
7,674649001,Dress,5.334804
8,765545001,Sweater,5.307489
9,687635018,Shorts,5.292216
